# 05 · Project A: Subscriptions & Entertainment over time

**Question:** how does Subscriptions & Entertainment spending in the Prism data move month to month, and does it
track national recreation spending and prices?

Lecture 4 asked each group to choose one category for an end-of-quarter deep dive compared with macroeconomic
trends from FRED; Lecture 5 laid out the steps: define eligible transactions, aggregate monthly count and value,
check coverage, compare with a justified external series.

- **Eligible:** debits the final model assigns to the category with a score above the review cutoff
- **Groups:** streaming & subscriptions, events & games, gambling, other (from the matching pre-tag rule)
- **Gambling** is reported separately and every series is also given without it
- **FRED:** `DRCARC1Q027SBEA` (PCE recreation services, quarterly), `CUUR0000SERA02` (CPI cable, satellite and
  live streaming TV), `CPIRECSL` (CPI recreation)

> Runs on DataHub. Cells show aggregate tables and charts only; files with memo text (errors, examples, predictions) stay in `outputs/` and are never printed here or committed.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
CONFIG = os.environ.get("CAPSTONE_CONFIG", "config.yaml")

import pandas as pd
from IPython.display import Image, display
from src.config import load_config
from src.main import run
OUT = load_config(CONFIG)["paths"]["out_dir"]
pd.set_option("display.max_colwidth", 80)

In [ ]:
m = run(CONFIG, "monthly")
m["monthly"].get("theme")

In [ ]:
display(Image(OUT / "theme_monthly.png"))

In [ ]:
pd.read_csv(OUT / "theme_monthly.csv").head(24)

In [ ]:
pd.read_csv(OUT / "theme_quarterly.csv")

## Comparison with FRED

In [ ]:
fred_path = OUT / "fred_series.csv"
if fred_path.exists():
    fred = pd.read_csv(fred_path, parse_dates=["date"]).set_index("date")
    q = pd.read_csv(OUT / "theme_quarterly.csv")
    q = q[q.series == "all"].assign(date=lambda d: pd.PeriodIndex(d.quarter, freq="Q").start_time).set_index("date")
    both = q[["dollars"]].join(fred["DRCARC1Q027SBEA"], how="inner").pct_change().dropna()
    display(both.rename(columns={"dollars": "Prism S&E dollars, q/q", "DRCARC1Q027SBEA": "PCE recreation, q/q"}))
    print("correlation of quarterly growth:", round(both.corr().iloc[0, 1], 2), "on", len(both), "quarters")
else:
    print("fred_series.csv not found: download the series from FRED into outputs/ (see README)")

Compare growth rates and turning points, not levels: the Prism sample covers its accounts, not the
US population, and the FRED series are seasonally adjusted while ours are not. Only accepted predictions are
counted, so check the theme coverage numbers above before reading a trend.